In [1]:
from __future__ import annotations

import sys
from pathlib import Path
import math
import warnings

PROJECT_ROOT = Path("/home/playdata2/final_pj/energy-platform")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.tsa.seasonal import STL

from scripts.eda.stl.eda_stl_electric import build_input_series
from scripts.pipeline.preprocess import fetch_joined_data

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

P_VALUE_THRESHOLD = 0.05
YEARS = [2018, 2019, 2020, 2021, 2022, 2023]
PERIOD = 24 * 7
SEASONAL = 13
SIGMA = 3

CONSUMPTION_REPS = [
    "H1.Z13", "H1.Z21", "H1.Z24", "H1.Z12", "H2.Z66", "H4.Z51", "H2.Z70", "H2.Z351",
    "H2.Z61", "H2.Z36", "H2.Z62", "H2.Z64", "H3.Z43", "H3.Z44", "H3.Z48", "H4.Z50",
    "H1.Z10", "H1.Z16", "H1.Z18", "H1.Z19", "H1.Z23", "H1.Z26", "H1.Z27", "H2.Z65",
    "H2.Z68", "H2.Z69", "H2.ZE65", "H2.ZE74", "H3.Z40", "H3.Z41", "H3.Z42", "H3.Z45",
    "H3.Z46", "H3.Z47", "H3.Z71", "H2.T.Z31", "H2.T.Z32", "H2.Z351", "H2.Z361",
    "H4.Z50", "H4.ZE50", "H4.Z51", "H4.ZE51",
]

PRODUCTION_REPS = ["V.Z84", "H1.Z20"]

THERMAL_METERS = [
    "V.K21", "H1.K11", "H1.K12", "H1.K14", "H1.K15", "H1.K16", "H2.K21", "H1.W11", "H1.W12"
]

In [2]:
def unique_preserve_order(values: list[str]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for value in values:
        if value in seen:
            continue
        seen.add(value)
        ordered.append(value)
    return ordered


CONSUMPTION_REPS = unique_preserve_order(CONSUMPTION_REPS)
PRODUCTION_REPS = unique_preserve_order(PRODUCTION_REPS)
THERMAL_METERS = unique_preserve_order(THERMAL_METERS)

METER_FEATURES: list[tuple[str, str, str]] = []
for meter_urn in CONSUMPTION_REPS:
    for feature in ["P", "PF", "U1"]:
        METER_FEATURES.append((meter_urn, "electric_consumption", feature))
for meter_urn in PRODUCTION_REPS:
    for feature in ["P", "PF", "U1"]:
        METER_FEATURES.append((meter_urn, "electric_production", feature))
for meter_urn in THERMAL_METERS:
    for feature in ["P", "qv", "Tdiff"]:
        METER_FEATURES.append((meter_urn, "thermal", feature))

RAW_CACHE: dict[str, pd.DataFrame] = {}
DETAIL_CACHE: dict[tuple[str, str], pd.DataFrame] = {}


def get_raw_meter_data(meter_urn: str) -> pd.DataFrame:
    if meter_urn not in RAW_CACHE:
        df = fetch_joined_data(meter_urn).copy()
        df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
        RAW_CACHE[meter_urn] = df
    return RAW_CACHE[meter_urn].copy()


def build_stl_detail(meter_urn: str, feature: str) -> pd.DataFrame:
    cache_key = (meter_urn, feature)
    if cache_key in DETAIL_CACHE:
        return DETAIL_CACHE[cache_key].copy()

    df = get_raw_meter_data(meter_urn)
    if feature not in df.columns:
        raise ValueError(f"{meter_urn}: feature '{feature}' column missing")

    series = build_input_series(df, feature)
    series = pd.to_numeric(series, errors="coerce").copy()
    series = series.interpolate(method="linear", limit=24)
    stl_input = series.dropna()
    if len(stl_input) < PERIOD:
        raise ValueError(f"{meter_urn}-{feature}: not enough points for STL ({len(stl_input)})")

    result = STL(stl_input, period=PERIOD, seasonal=SEASONAL).fit()
    residual = pd.to_numeric(result.resid, errors="coerce")
    mean = residual.mean()
    std = residual.std()
    upper = mean + SIGMA * std
    lower = mean - SIGMA * std
    anomaly_mask = (residual > upper) | (residual < lower)

    detail = pd.DataFrame(
        {
            "meter_urn": meter_urn,
            "feature": feature,
            "ts": residual.index,
            "observed": result.observed.values,
            "trend": result.trend.values,
            "seasonal": result.seasonal.values,
            "residual": residual.values,
            "upper": upper,
            "lower": lower,
            "is_anomaly": anomaly_mask.values,
        }
    )
    detail["ts"] = pd.to_datetime(detail["ts"], utc=True, errors="coerce")
    DETAIL_CACHE[cache_key] = detail
    return detail.copy()


def format_annual_std(annual_std: pd.Series) -> str:
    parts: list[str] = []
    for year in YEARS:
        value = annual_std.get(year, np.nan)
        if pd.isna(value):
            parts.append(f"{year}=NaN")
        else:
            parts.append(f"{year}={value:.3f}")
    return ", ".join(parts)


print(f"consumption reps (deduped): {len(CONSUMPTION_REPS)}")
print(f"production reps: {len(PRODUCTION_REPS)}")
print(f"thermal meters: {len(THERMAL_METERS)}")
print(f"total meter-feature combinations: {len(METER_FEATURES)}")
print(f"STL params: period={PERIOD}, seasonal={SEASONAL}")


consumption reps (deduped): 40
production reps: 2
thermal meters: 9
total meter-feature combinations: 153
STL params: period=168, seasonal=13


In [3]:
RESULTS: list[dict[str, object]] = []


def summarize_homoscedasticity(meter_urn: str, group_name: str, feature: str) -> dict[str, object]:
    detail = build_stl_detail(meter_urn, feature)
    working = detail[["ts", "residual"]].copy()
    working["ts"] = pd.to_datetime(working["ts"], utc=True, errors="coerce")
    working["residual"] = pd.to_numeric(working["residual"], errors="coerce")
    working = working.dropna(subset=["ts", "residual"]).sort_values("ts").reset_index(drop=True)
    if len(working) < 10:
        raise ValueError(f"{meter_urn}-{feature}: too few residual points ({len(working)})")

    # User instruction: auxiliary regression residual^2 ~ time_index.
    exog = sm.add_constant(np.arange(len(working), dtype=float))
    lm_stat, lm_pvalue, f_stat, f_pvalue = het_breuschpagan(working["residual"].values, exog)

    annual_std = working.groupby(working["ts"].dt.year)["residual"].std().reindex(YEARS)
    valid_std = annual_std.dropna()
    positive_std = valid_std[valid_std > 0]
    if positive_std.empty:
        variance_ratio = np.nan
    else:
        variance_ratio = float(valid_std.max() / positive_std.min())

    if valid_std.empty:
        focus_years: list[int] = []
    else:
        max_std = float(valid_std.max())
        focus_years = [int(year) for year, value in valid_std.items() if pd.notna(value) and float(value) >= max_std * 0.9]

    status = "PASS" if lm_pvalue >= P_VALUE_THRESHOLD else "FAIL"
    annual_std_text = format_annual_std(annual_std)
    ratio_text = f"{variance_ratio:.2f}배" if pd.notna(variance_ratio) and math.isfinite(variance_ratio) else "계산 불가"
    focus_text = ", ".join(str(year) for year in focus_years) if focus_years else "-"
    status_text = "등분산 PASS" if status == "PASS" else "이분산 FAIL"

    print(f"[{meter_urn} - {feature}]")
    print(f"  p-value: {lm_pvalue:.6f}  -> {status_text}")
    print(f"  연도별 std: {annual_std_text}")
    print(f"  분산 최대/최소 비율: {ratio_text}")
    print(f"  이분산 집중 연도: {focus_text}")
    print()

    return {
        "group": group_name,
        "meter": meter_urn,
        "feature": feature,
        "p_value": float(lm_pvalue),
        "bp_lm_stat": float(lm_stat),
        "bp_f_stat": float(f_stat),
        "bp_f_pvalue": float(f_pvalue),
        "status": status,
        "variance_ratio": variance_ratio,
        "focus_years": focus_text,
        **{f"std_{year}": annual_std.get(year, np.nan) for year in YEARS},
    }


for meter_urn, group_name, feature in METER_FEATURES:
    try:
        RESULTS.append(summarize_homoscedasticity(meter_urn, group_name, feature))
    except Exception as exc:
        print(f"[{meter_urn} - {feature}]")
        print(f"  실패: {exc}")
        print()
        RESULTS.append(
            {
                "group": group_name,
                "meter": meter_urn,
                "feature": feature,
                "p_value": np.nan,
                "bp_lm_stat": np.nan,
                "bp_f_stat": np.nan,
                "bp_f_pvalue": np.nan,
                "status": "SKIP",
                "variance_ratio": np.nan,
                "focus_years": "-",
                **{f"std_{year}": np.nan for year in YEARS},
            }
        )


[H1.Z13 - P]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=2902.951, 2019=2765.311, 2020=2817.065, 2021=2236.941, 2022=2176.025, 2023=1321.632
  분산 최대/최소 비율: 2.20배
  이분산 집중 연도: 2018, 2019, 2020



[H1.Z13 - PF]
  p-value: 0.095327  -> 등분산 PASS
  연도별 std: 2018=0.030, 2019=0.029, 2020=0.032, 2021=0.028, 2022=0.036, 2023=0.029
  분산 최대/최소 비율: 1.31배
  이분산 집중 연도: 2022



[H1.Z13 - U1]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=1.138, 2019=1.123, 2020=1.078, 2021=1.045, 2022=1.061, 2023=1.044
  분산 최대/최소 비율: 1.09배
  이분산 집중 연도: 2018, 2019, 2020, 2021, 2022, 2023



[H1.Z21 - P]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=685.754, 2019=872.990, 2020=695.329, 2021=521.001, 2022=441.279, 2023=259.024
  분산 최대/최소 비율: 3.37배
  이분산 집중 연도: 2019



[H1.Z21 - PF]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=0.181, 2019=0.043, 2020=0.031, 2021=0.130, 2022=0.101, 2023=0.113
  분산 최대/최소 비율: 5.91배
  이분산 집중 연도: 2018



...

[H1.W11 - qv]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=3.545, 2019=2.188, 2020=3.608, 2021=2.121, 2022=2.966, 2023=8.934
  분산 최대/최소 비율: 4.21배
  이분산 집중 연도: 2023



[H1.W11 - Tdiff]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=1235.240, 2019=1153.938, 2020=1108.436, 2021=1210.782, 2022=970.285, 2023=4361.165
  분산 최대/최소 비율: 4.49배
  이분산 집중 연도: 2023



[H1.W12 - P]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=56141.335, 2019=59325.434, 2020=55855.734, 2021=62874.219, 2022=52471.715, 2023=67538.230
  분산 최대/최소 비율: 1.29배
  이분산 집중 연도: 2021, 2023



[H1.W12 - qv]
  p-value: 0.000000  -> 이분산 FAIL
  연도별 std: 2018=2.517, 2019=3.421, 2020=3.009, 2021=3.445, 2022=3.081, 2023=3.982
  분산 최대/최소 비율: 1.58배
  이분산 집중 연도: 2023



[H1.W12 - Tdiff]
  p-value: 0.577592  -> 등분산 PASS
  연도별 std: 2018=3046.704, 2019=2704.489, 2020=2604.686, 2021=2847.012, 2022=2430.885, 2023=3153.874
  분산 최대/최소 비율: 1.30배
  이분산 집중 연도: 2018, 2021, 2023



In [4]:
results_df = pd.DataFrame(RESULTS)
summary_df = results_df.copy()
summary_df["p-value"] = summary_df["p_value"].map(lambda x: f"{x:.6f}" if pd.notna(x) else "NaN")
summary_df["판정"] = summary_df["status"]
summary_df["분산비율"] = summary_df["variance_ratio"].map(lambda x: f"{x:.2f}배" if pd.notna(x) and math.isfinite(x) else "NaN")
summary_df["이분산연도"] = summary_df["focus_years"]
summary_df = summary_df[["meter", "feature", "p-value", "판정", "분산비율", "이분산연도"]].sort_values(["meter", "feature"]).reset_index(drop=True)

print("전체 결과 요약")
print("-" * 60)
display(summary_df)

valid_mask = results_df["status"].isin(["PASS", "FAIL"])
total_count = int(valid_mask.sum())
pass_count = int((results_df["status"] == "PASS").sum())
fail_count = int((results_df["status"] == "FAIL").sum())
skip_count = int((results_df["status"] == "SKIP").sum())
pass_ratio = (pass_count / total_count * 100) if total_count else float("nan")
fail_ratio = (fail_count / total_count * 100) if total_count else float("nan")

print(f"전체: {total_count}개 조합")
print(f"PASS: {pass_count}개 ({pass_ratio:.2f}%)")
print(f"FAIL: {fail_count}개 ({fail_ratio:.2f}%)")
print(f"SKIP: {skip_count}개")


전체 결과 요약
------------------------------------------------------------


,meter,feature,p-value,판정,분산비율,이분산연도
0,H1.K11,P,0.000000,FAIL,2.89배,2023
1,H1.K11,Tdiff,0.000000,FAIL,1.77배,2021
2,H1.K11,qv,0.000000,FAIL,3.40배,2023
3,H1.K12,P,0.000000,FAIL,273.83배,"2019, 2020"
4,H1.K12,Tdiff,0.000000,FAIL,5.65배,"2019, 2020, 2021"
5,H1.K12,qv,0.000000,FAIL,763.46배,"2019, 2020, 2021"
6,H1.K14,P,0.000085,FAIL,1.39배,"2022, 2023"
7,H1.K14,Tdiff,0.000000,FAIL,1.92배,"2018, 2019"
8,H1.K14,qv,0.000000,FAIL,2.21배,2019
9,H1.K15,P,0.000200,FAIL,1.52배,2023


전체: 152개 조합
PASS: 17개 (11.18%)
FAIL: 135개 (88.82%)
SKIP: 1개


In [5]:
print("[판단 기준]\n")
print("등분산 PASS:")
print("  -> 잔차 안정적")
print("  -> 정규분포 가정 가능")
print("  -> 고정 임계값(3sigma) 신뢰 가능")
print("  -> 회귀 기반 이상탐지 적용 가능\n")

print("이분산 FAIL:")
print("  -> 원인 파악 필요\n")
print("  원인 1: 운영 조건 변화")
print("    예: H1.W11 2023년 난방 현대화")
print("    -> is_regime_change 처리 후 재검정\n")
print("  원인 2: 계절적 분산 변화")
print("    -> 계절별 임계값 분리 적용 검토\n")
print("  원인 3: 이상 구간 포함")
print("    -> 해당 구간 NaN 처리 후 재검정\n")
print("  FAIL 비율이 높으면:")
print("    -> LSTM-AE (복원 기반) 모델이 더 적합")
print("    -> 고정 임계값 대신 동적 임계값 검토")


[판단 기준]

등분산 PASS:
  -> 잔차 안정적
  -> 정규분포 가정 가능
  -> 고정 임계값(3sigma) 신뢰 가능
  -> 회귀 기반 이상탐지 적용 가능

이분산 FAIL:
  -> 원인 파악 필요

  원인 1: 운영 조건 변화
    예: H1.W11 2023년 난방 현대화
    -> is_regime_change 처리 후 재검정

  원인 2: 계절적 분산 변화
    -> 계절별 임계값 분리 적용 검토

  원인 3: 이상 구간 포함
    -> 해당 구간 NaN 처리 후 재검정

  FAIL 비율이 높으면:
    -> LSTM-AE (복원 기반) 모델이 더 적합
    -> 고정 임계값 대신 동적 임계값 검토
